__Diese Abfrage bezieht sich auf die Maximalwerte pro Aufnahme, die in der Tabelle model_inference_max_confidence gespeichert sind.__

Die Teilabfrage sollte für alle Inferenzen >= 0,4 für die folgenden Arten 
enthalten (ähnlich wie die Falkenabfrage, nur diesmal mit Schwellenwert). 
    
Die mit Fragezeichen gekennzeichneten Arten sind möglicherweise nicht im Modell 
bzw. in den Inferenzen >=0,1 enthalten. 
    
Weitere Arten sind möglicherweise nicht mit Konfidenzwerten >= 0,4 vertreten. 
Die entsprechenden Tabellen bleiben dann leer.

* Tettigonia viridissima
* Tettigonia cantans
* Tettigonia caudata
* Acheta domesticus
* Eumodicogryllus bordigalensis
* Gryllodes sigillatus (?)
* Gryllus bimaculatus
* Gryllus campestris
* Modicogryllus frontalis (?)
* Nemobius sylvestris
* Pteronemobius heydenii
* Oecanthus pellucens

## Connect to database
Follow the instruction in the Notebook [00_Setup_DB_Connection](./00_Setup_DB_Connection.ipynb).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Load the database username and password
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read the username and password
username = os.getenv("DB_USERNAME")
password = os.getenv("DB_PASSWORD")

DB_CONFIG = {
    'host': 'localhost',
    'port': 8432,  # the port opened using the ssh tunnel
    'user': username,
    'password': password,
    'database': 'ecomon'
}

instance_name = "validate"
model_name = 'BNP_v3_0_glob11k_prev3_cw5s_sz2s'

## Test the connection

In [ ]:
from bslquery import BioSoundsLabDB

# Initialize the database helper
db = BioSoundsLabDB(DB_CONFIG, verbose=True)

# Test the connection
db.test_connection()

## Fetch sites and years
This will return one entry for each combination of site prefix and year, and the count of records.

In [ ]:
# List all sites and years combinations
site_years = db.sites_years()
site_years

## Fetch species labels
Just to make sure the names are in the database.

In [ ]:
species = [
    'Tettigonia viridissima',
    'Tettigonia cantans',
    'Tettigonia caudata',
    'Acheta domesticus',
    'Eumodicogryllus bordigalensis',
    'Gryllodes sigillatus',  # (?)
    'Gryllus bimaculatus',
    'Gryllus campestris',
    'Modicogryllus frontalis',  # (?)
    'Nemobius sylvestris',
    'Pteronemobius heydenii',
    'Oecanthus pellucens'
]
species_labels = []
for spc in species:
    species_labels.append({
        'label_id': db.fetch_label_id_by_species_name(spc),
        'ScientificName_IOC10.1': spc
    })
species_labels

## Fetch model id
Just to make sure the model is in the database.

In [ ]:
model_id = db.fetch_model_id_by_model_name(model_name)
model_id

## Fetch inferences

In [ ]:
import pandas as pd

def prepare_output_data(inferences):
    # Prepare output data
    output_df = pd.DataFrame(inferences)
    # Convert start_time and end_time to integers
    output_df['start_time'] = output_df['start_time'].astype(int)
    output_df['end_time'] = output_df['end_time'].astype(int)
    
    # Insert (empty) extra columns 
    columns_to_insert = [
        (7, 'ManualValidation'),
        (6, 'VocalizationTypeCode'),
        (5, 'Manual_Trigger'),
        (4, 'Manual_OtherSounds_5s'),
        (3, 'Notes_on_complete_snippet_10_15s'),
    ]
    
    # Sort by position_from_end in ascending order
    columns_to_insert.sort(key=lambda x: x[0])
    
    # Insert columns in order
    for position_from_end, new_column_name in columns_to_insert:
        insert_index = len(output_df.columns) - position_from_end
        output_df.insert(loc=insert_index, column=new_column_name, value=None)

    return output_df

___Before executing the next cell, create a folder where the spreadsheets with the data will be stored.___

The folder should be called: examples/data (this folder will not be committed).

In [ ]:
def save_output_data(output_df, species_name, instance_name):
    # export as excel
    filename = f"MaxInf_{species_name}_{instance_name}.xlsx".replace(' ', '_')
    output_df.to_excel(
        f"data/{filename}",
        sheet_name='Data',
        index=False,
        header=True,
    )

In [ ]:
for species_current in species_labels:
    species_name = species_current['ScientificName_IOC10.1']
    inferences = list()
    for current_site_year in site_years:
        new = db.fetch_max_pro_record(
            species_name, model_name, current_site_year['prefix'], current_site_year['year'],
            min_confidence=0.4
        )
        if new is not None:
            inferences += new
    if len(inferences) > 0:
        output_df = prepare_output_data(inferences)
    else:
        output_df = pd.DataFrame()
    save_output_data(output_df, species_name, instance_name)
print("Done")